# Lesson 7: Perturb a model

**You will learn:** how to silence or scale units inside a model during a run, and how to undo the change.
**Requires:** CPU only, no downloads, under a minute.

In a lesion or inactivation study, a neuroscientist silences part of the brain and observes what changes. The same can be done with a model. Three objects describe a perturbation:

| Object | Meaning |
| --- | --- |
| `Selection` | Which units: a layer path and, optionally, unit indices. |
| `Perturbation` | How to change them: `'zero'`, `'scale'` or `'replace'`. |
| `StateChange` | The request: apply a perturbation to a selection, or undo one (`kind='reset'`). |

A `BrainScoreModel` applies perturbations through its `intervention` part. `build_pytorch_ablation_fn` builds one for any PyTorch network using forward hooks.

In [1]:
import warnings
warnings.filterwarnings('ignore')  # keep the lesson output short

In [2]:
import torch
from brainscore_core import BrainScoreModel
from brainscore.perturbation import build_pytorch_ablation_fn

torch.manual_seed(0)
net = torch.nn.Sequential(torch.nn.Linear(4, 8), torch.nn.ReLU(), torch.nn.Linear(8, 2)).eval()
x = torch.ones(1, 4)

model = BrainScoreModel(identifier='toy-mlp', model=net, intervention=build_pytorch_ablation_fn(net))
print('inputs: ', model.in_channels)
print('outputs:', model.out_channels)

baseline = net(x).detach()
baseline

inputs:  {'lesion', 'stimulation'}
outputs: {'perturbation'}


tensor([[-0.1394, -0.1776]])

## Silence half of the hidden units

In [3]:
from brainscore_core import Perturbation, Selection, StateChange

applied = model.process(StateChange(
    kind='ablation',
    target=Selection(layer='0', indices=[0, 1, 2, 3]),
    perturbation=Perturbation(kind='zero'),
))
print('handle:', applied.handle_id)
net(x).detach()

handle: 79be8cb4-b104-4299-82a7-0abf07520fc6


tensor([[-0.1217, -0.1764]])

The output changed because four of the eight hidden units now output zero. The returned `PerturbationApplied` record carries a `handle_id` that identifies this change.

## Undo the change

Undo one perturbation by its handle, or undo all of them with `model.reset()`.

In [4]:
model.process(StateChange(kind='reset', handle_id=applied.handle_id))
torch.allclose(net(x), baseline)

True

## Scale instead of silence

In [5]:
model.process(StateChange(kind='ablation', target=Selection(layer='0'),
                          perturbation=Perturbation(kind='scale', scale=0.5)))
print(net(x).detach())
model.reset()
torch.allclose(net(x), baseline)

tensor([[-0.0368, -0.2259]])


True

## Perturbations inside a benchmark

A perturbation is also an input event, on a `lesion` or `stimulation` channel, so a benchmark can deliver it through a session just like a stimulus. The `apply_state_change(subject, state_change)` helper does this for one change. The usual pattern in a study is baseline, perturbed, restored, as shown in [Lesson 10](10_run_an_experiment.ipynb).

## Summary

- `Selection` says which units; `Perturbation` says how; `StateChange` applies or undoes it.
- The `intervention` part does the work; `build_pytorch_ablation_fn` covers PyTorch networks.
- Always undo perturbations (`reset()`) before the next independent run.

**Next:** [Lesson 8](08_choose_units.ipynb) chooses *which* units to perturb in a principled way.